<img src="https://raw.githubusercontent.com/drdave-teaching/OPIM5509-notebooks/main/_banners/opim5509_banner.svg" width="100%" alt="OPIM 5509 banner"/>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/drdave-teaching/OPIM5509Files/blob/main/OPIM5509_Module5_Files/notebooks/M5_2b_Word_Embeddings.ipynb)

🔴
<!-- 🎙 DAVE TALKING POINTS — M5 · 7 — Tokenize, pad, and the Embedding layer (count its parameters)
- Say the line first: M5.1 threw word order away. M5.2 KEEPS it - and step one is turning each word into a vector.
- The picture: one-hot = 10,000 columns of mostly zeros, fixed by hand. An embedding = 8 (or 100) dense numbers per word, LEARNED. Similar words end up close together - king - man + woman lands near queen.
- Two ways to get one: learn it along with your task (this video) or borrow a pretrained one (next video).
- Embedding(1000, 64) is just a lookup table: 1,000 words x 64 numbers = 64,000 parameters.
- IMDB comes pre-tokenized in Keras: keep the top 10,000 words, and pad_sequences cuts every review to its first 20 words (short ones get zeros in front).
- Input(20) -> Embedding(10000, 8) = 80,000 -> Flatten 20 x 8 = 160 -> Dense(1) = 161. Total 80,161. The Input layer is the Keras 3 way - input_length is ignored now.
- Validation accuracy peaks at 0.76 (epoch 7) from 20 words per review; best validation loss 0.498 at epoch 5. But Flatten treats each position separately: 'this movie is shit' vs 'this movie is the shit' - recurrent layers fix that in video 9.
-->

# Word embeddings: from one-hot to learned vectors

--------------------------------------------------
**Dr. Dave Wanik - University of Connecticut**

The basic ways to turn words into numbers, from the obvious to the powerful: **one-hot** (one column per word), then **learned embeddings** (a short vector per word, flattened into a Dense network) - and what flattening can't do. The geometry of GloVe is in M5_2a; reading words *in order* is M5_2c.

Adapted from Chollet, *Deep Learning with Python*, Chapter 6, Section 1.

In [1]:
import keras
keras.__version__

'3.15.1'

In [2]:
# seed everything: same numbers on my screen and yours
# (and we re-seed right before every model build, so model 2 is as reproducible as model 1)
import keras
import tensorflow as tf

keras.utils.set_random_seed(5509)
tf.config.experimental.enable_op_determinism()

In [3]:
# The data below is downloaded from public sources (Stanford), so no Google Drive is needed.
# (Only mount Drive if you want to save your trained model there:)
# from google.colab import drive
# drive.mount('/content/drive')

![word embeddings vs. one hot encoding](https://s3.amazonaws.com/book.keras.io/img/ch6/word_embeddings.png)

There are two ways to obtain word embeddings:

* Learn word embeddings jointly with the main task you care about (e.g. document classification or sentiment prediction).
In this setup, you would start with random word vectors, then learn your word vectors in the same way that you learn the weights of a neural network.
* Load into your model word embeddings that were pre-computed using a different machine learning task than the one you are trying to solve.
These are called "pre-trained word embeddings".

Let's take a look at both.

## Learning word embeddings with the `Embedding` layer


The simplest way to associate a dense vector to a word would be to pick the vector at random. The problem with this approach is that the
resulting embedding space would have no structure: for instance, the words "accurate" and "exact" may end up with completely different
embeddings, even though they are interchangeable in most sentences. It would be very difficult for a deep neural network to make sense of
such a noisy, unstructured embedding space.

To get a bit more abstract: the geometric relationships between word vectors should reflect the semantic relationships between these words.
Word embeddings are meant to map human language into a geometric space. For instance, in a reasonable embedding space, we would expect
synonyms to be embedded into similar word vectors, and in general we would expect the geometric distance (e.g. L2 distance) between any two
word vectors to relate to the semantic distance of the associated words (words meaning very different things would be embedded to points
far away from each other, while related words would be closer). Even beyond mere distance, we may want specific __directions__ in the
embedding space to be meaningful.

[...]


In real-world word embedding spaces, common examples of meaningful geometric transformations are "gender vectors" and "plural vector". For
instance, by adding a "female vector" to the vector "king", one obtain the vector "queen". By adding a "plural vector", one obtain "kings".
Word embedding spaces typically feature thousands of such interpretable and potentially useful vectors.

Is there some "ideal" word embedding space that would perfectly map human language and could be used for any natural language processing
task? Possibly, but in any case, we have yet to compute anything of the sort. Also, there isn't such a thing as "human language", there are
many different languages and they are not isomorphic, as a language is the reflection of a specific culture and a specific context. But more
pragmatically, what makes a good word embedding space depends heavily on your task: the perfect word embedding space for an
English-language movie review sentiment analysis model may look very different from the perfect embedding space for an English-language
legal document classification model, because the importance of certain semantic relationships varies from task to task.

It is thus reasonable to __learn__ a new embedding space with every new task. Thankfully, backpropagation makes this really easy, and Keras makes it
even easier. It's just about learning the weights of a layer: the `Embedding` layer.

In [4]:
from keras.layers import Embedding

# The Embedding layer takes at least two arguments:
# the number of possible tokens, here 1000 (1 + maximum word index),
# and the dimensionality of the embeddings, here 64.
embedding_layer = Embedding(1000, 64)


The `Embedding` layer is best understood as a dictionary mapping integer indices (which stand for specific words) to dense vectors. It takes
as input integers, it looks up these integers into an internal dictionary, and it returns the associated vectors. It's effectively a dictionary lookup.


The `Embedding` layer takes as input a 2D tensor of integers, of shape `(samples, sequence_length)`, where each entry is a sequence of
integers. It can embed sequences of variable lengths, so for instance we could feed into our embedding layer above batches that could have
shapes `(32, 10)` (batch of 32 sequences of length 10) or `(64, 15)` (batch of 64 sequences of length 15). All sequences in a batch must
have the same length, though (since we need to pack them into a single tensor), so sequences that are shorter than others should be padded
with zeros, and sequences that are longer should be truncated.

This layer returns a 3D floating point tensor, of shape `(samples, sequence_length, embedding_dimensionality)`. Such a 3D tensor can then
be processed by a RNN layer or a 1D convolution layer (both will be introduced in the next sections).

When you instantiate an `Embedding` layer, its weights (its internal dictionary of token vectors) are initially random, just like with any
other layer. During training, these word vectors will be gradually adjusted via backpropagation, structuring the space into something that the
downstream model can exploit. Once fully trained, your embedding space will show a lot of structure -- a kind of structure specialized for
the specific problem you were training your model for.

Let's apply this idea to the IMDB movie review sentiment prediction task that you are already familiar with. Let's quickly prepare
the data. We will restrict the movie reviews to the top 10,000 most common words (like we did the first time we worked with this dataset),
and cut the reviews after only 20 words. Our network will simply learn 8-dimensional embeddings for each of the 10,000 words, turn the
input integer sequences (2D integer tensor) into embedded sequences (3D float tensor), flatten the tensor to 2D, and train a single `Dense`
layer on top for classification.

In [5]:
from keras.datasets import imdb
from keras import preprocessing
from keras.utils import pad_sequences

# Number of words to consider as features
max_features = 10000
# cut every review after N words - here N = maxlen = 20
# (careful: pad_sequences keeps the LAST 20 words by default, not the first 20)
maxlen = 20

# Load the data as lists of integers.
(x_train, y_train), (x_test, y_test) = imdb.load_data(num_words=max_features)

# This turns our lists of integers
# into a 2D integer tensor of shape `(samples, maxlen)`
x_train = pad_sequences(x_train, maxlen=maxlen)
x_test = pad_sequences(x_test, maxlen=maxlen)

### What does the model actually see?
Keras ships IMDB already turned into numbers - every word is replaced by its rank (1 = most common). Let's turn two reviews back into words and look at the label and the 20 numbers that go into the model.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Keras shifts every word's rank by 3: 0 = padding, 1 = start of review, 2 = unknown word
word_index = imdb.get_word_index()
index_word = {0: '<pad>', 1: '<start>', 2: '<unk>'}
for word, rank in word_index.items():
    index_word[rank + 3] = word
label_name = {0: 'negative', 1: 'positive'}

peek_words = []          # we'll reuse these two reviews later (GloVe!)
fig, axes = plt.subplots(2, 1, figsize=(14, 7))
for k in [0, 1]:
    words = []
    for number in x_train[k]:
        words.append(index_word.get(number, '?'))
    peek_words.append(words)
    print('review', k, '| label:', y_train[k], '(' + label_name[y_train[k]] + ')')
    print('   words :', ' '.join(words))
    print('   vector:', x_train[k])
    axes[k].bar(range(1, maxlen + 1), x_train[k])
    axes[k].set_xticks(range(1, maxlen + 1))
    axes[k].set_xticklabels(words, rotation=60)
    axes[k].set_ylabel('word number')
    axes[k].set_title('review ' + str(k) + ' - label ' + str(y_train[k]) + ' (' + label_name[y_train[k]] + ')')
plt.tight_layout()
plt.show()

# THIS is what the model sees: 20 integers. The bar height is just a word's rank - it means nothing yet.
# The Embedding layer is what turns each integer into a useful vector.

### Step 1: one-hot encoding - the obvious way to turn words into numbers
Give every word its own column, and put a 1 in the column of the word you see. One review of 20 words becomes a 20 x 10,000 grid of zeros with twenty 1s. It works - and you can one-hot an entire data set - but look at the size.

In [ ]:
# ONE word, one-hot: a row of 10,000 numbers - all 0, except a single 1 in that word's column
word = 'lovely'
col = word_index[word] + 3                 # + 3 because Keras shifts every rank by 3
vector = np.zeros(max_features)
vector[col] = 1

plt.figure(figsize=(14, 2.8))
plt.plot(range(max_features), vector, color='tab:blue')
plt.annotate(f"'{word}' -> a single 1 in column {col}", (col, 1), xytext=(col + 400, 0.75),
             arrowprops={'arrowstyle': '->'}, fontsize=11)
plt.ylim(-0.1, 1.2)
plt.xlabel('vocabulary column (0 - 9,999)')
plt.title(f"one-hot for the word '{word}': 10,000 numbers - 9,999 zeros and one 1")
plt.show()
print('numbers:', vector.size, '| ones:', int(vector.sum()), '| zeros:', int((vector == 0).sum()))
# a whole review is just 20 of these rows stacked up - one per word (next cell)

In [ ]:
# one-hot review 0: one row per word, one column per vocabulary word
one_hot = np.zeros((maxlen, max_features))
for slot in range(maxlen):
    one_hot[slot, x_train[0][slot]] = 1
print('one review, one-hot:', one_hot.shape, '=', one_hot.size, 'numbers -', int(one_hot.sum()), 'of them are 1s')
print('all 25,000 training reviews would be', f'{25000 * one_hot.size:,}', 'numbers')

# show only the columns this review actually uses (every other column is all 0s)
used_cols = sorted(set(x_train[0]))
col_labels = []
for col in used_cols:
    col_labels.append(index_word.get(col, '?') + ' (#' + str(col) + ')')
plt.figure(figsize=(14, 7))
plt.imshow(one_hot[:, used_cols], cmap='Greys', vmin=0, vmax=1.4, aspect='auto')
for slot in range(maxlen):
    plt.text(used_cols.index(x_train[0][slot]), slot, '1', ha='center', va='center', color='white', fontweight='bold')
plt.yticks(range(maxlen), peek_words[0])
plt.xticks(range(len(used_cols)), col_labels, rotation=60, ha='right')
plt.ylabel('word slot (1 - 20)')
plt.title(f'review 0, one-hot: every row has a single 1, in its word\'s column\n'
          f'(showing the {len(used_cols)} columns this review uses - the other {max_features - len(used_cols):,} columns are all 0)')
plt.tight_layout()
plt.show()

### Step 2: one-hot a whole data set (bag of words, the M5.1 way) - a strong baseline!
Squash each review's one-hot grid into ONE row: a 1 if the word appears anywhere in the review. That's 10,000 columns per review, word order thrown away - and a plain Dense network on it is hard to beat. We use the **full** reviews here, not just 20 words.

In [ ]:
# multi-hot: one row per review, a 1 in the column of every word it uses (order thrown away)
(full_train, y_full_train), (full_test, y_full_test) = imdb.load_data(num_words=max_features)

def multi_hot(reviews):
    out = np.zeros((len(reviews), max_features), dtype='float32')
    for r in range(len(reviews)):
        for number in reviews[r]:
            out[r, number] = 1.0
    return out

X_hot_train = multi_hot(full_train)
X_hot_test = multi_hot(full_test)
print('one-hot data set:', X_hot_train.shape, '- one row per review, one column per word')

In [ ]:
# what multi-hot does to our two reviews: a 1 in the column of every word used (order and repeats thrown away)
for k in [0, 1]:
    row = multi_hot([full_train[k]])[0]
    ones = np.nonzero(row)[0]
    words = []
    for col in ones:
        words.append(index_word.get(col, '?'))
    print('review', k, '(' + label_name[y_full_train[k]] + '):', int(row.sum()), 'ones out of', max_features, 'columns')
    print('   the 1s are in the columns for:', ', '.join(words[0:30]), '...')

In [ ]:
from keras.models import Sequential
from keras.layers import Input, Dense
from keras.callbacks import EarlyStopping

keras.utils.set_random_seed(5509)   # re-seed right before the build: same starting weights every run
model = Sequential()
model.add(Input(shape=(max_features,)))
model.add(Dense(16, activation='relu'))
model.add(Dense(16, activation='relu'))
model.add(Dense(1, activation='sigmoid'))
model.compile(optimizer='rmsprop', loss='binary_crossentropy', metrics=['accuracy'])
model.summary()

early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
history = model.fit(X_hot_train, y_full_train,
                    epochs=50,
                    batch_size=512,
                    validation_split=0.2,
                    callbacks=[early_stop],
                    verbose=0)

# loss curve - always look at this after a fit!
loss = history.history['loss']
val_loss = history.history['val_loss']
epochs = range(1, len(loss) + 1)
plt.plot(epochs, loss, 'bo', label='Training loss')
plt.plot(epochs, val_loss, 'b', label='Validation loss')
plt.title('Loss curve - one-hot (multi-hot) bag of words, full reviews')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.show()
print('epochs run:', len(loss), '| best validation loss:', round(min(val_loss), 4), 'at epoch', val_loss.index(min(val_loss)) + 1)

# the honest score: HELD-OUT test reviews
from sklearn.metrics import classification_report, f1_score
preds = (model.predict(X_hot_test, verbose=0).ravel() > 0.5).astype(int)
print(classification_report(y_full_test, preds, target_names=['negative', 'positive']))
print('one-hot bag of words (full reviews) - test macro F1:', round(f1_score(y_full_test, preds, average='macro'), 3))

### Step 3: embeddings - a short, dense vector per word instead of 10,000 columns
The one-hot model is strong, but it's 10,000 numbers per review and it has no idea that *great* and *excellent* mean nearly the same thing. An **Embedding** layer gives every word a short vector (8 numbers here) that the model *learns* - and words used the same way end up with similar vectors. Below: 20 words per review, embedded, flattened.

In [ ]:
from keras.models import Sequential
from keras.layers import Flatten, Dense
from keras.layers import Input
from keras.callbacks import EarlyStopping

keras.utils.set_random_seed(5509)   # re-seed right before the build: same starting weights every run
model = Sequential()
# We specify the maximum input length with an Input layer
# so we can later flatten the embedded inputs
model.add(Input(shape=(maxlen,)))
model.add(Embedding(10000, 8))
# After the Embedding layer,
# our activations have shape `(samples, maxlen, 8)`.

# We flatten the 3D tensor of embeddings
# into a 2D tensor of shape `(samples, maxlen * 8)`
model.add(Flatten())

# We add the classifier on top
model.add(Dense(1, activation='sigmoid'))
model.compile(optimizer='rmsprop', loss='binary_crossentropy', metrics=['accuracy'])
model.summary()

# stop when validation loss stops improving, and hand back the BEST epoch's weights
early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
history = model.fit(x_train, y_train,
                    epochs=50,
                    batch_size=32,
                    validation_split=0.2,
                    callbacks=[early_stop],
                    verbose=0)

# loss curve - always look at this after a fit!
loss = history.history['loss']
val_loss = history.history['val_loss']
epochs = range(1, len(loss) + 1)
plt.plot(epochs, loss, 'bo', label='Training loss')
plt.plot(epochs, val_loss, 'b', label='Validation loss')
plt.title('Loss curve - learned 8-number embeddings, 20 words')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.show()
print('epochs run:', len(loss), '| best validation loss:', round(min(val_loss), 4), 'at epoch', val_loss.index(min(val_loss)) + 1)

In [ ]:
# the honest score: HELD-OUT test reviews the model never saw while training
from sklearn.metrics import classification_report, f1_score
probs = model.predict(x_test, verbose=0).ravel()
preds = (probs > 0.5).astype(int)
print(classification_report(y_test, preds, target_names=['negative', 'positive']))
print('learned embedding (20 words) - test macro F1:', round(f1_score(y_test, preds, average='macro'), 3))

### Why Flatten is a clumsy way to read text: every word is locked to its slot
The Embedding layer turns each of the 20 words into 8 numbers - a 20 x 8 grid. `Flatten` then lays those 20 rows end to end: 160 numbers. The Dense layer gives each of the 160 its own weight, so it learns *"'excellent' in slot 20"* - a completely separate thing from *"'excellent' in slot 10"*. Same word, different slot, different weight. That's tough for a dense layer, and it's exactly what recurrent layers fix (video 9).

In [ ]:
# review 0 after the Embedding layer: 20 words x 8 numbers each
rows = model.layers[0](x_train[0:1]).numpy()[0]
flat = rows.reshape(-1)                       # what Flatten hands the Dense layer: 20 x 8 = 160 numbers
print('embedded review:', rows.shape, '-> flattened:', flat.shape)

fig, axes = plt.subplots(2, 1, figsize=(14, 8))
axes[0].imshow(rows.T, aspect='auto', cmap='coolwarm')
axes[0].set_xticks(range(maxlen))
axes[0].set_xticklabels(peek_words[0], rotation=60)
axes[0].set_ylabel('embedding number (1-8)')
axes[0].set_title('review 0 after the Embedding layer: one column of 8 numbers per word')
axes[1].bar(range(len(flat)), flat)
for slot in range(1, maxlen):
    axes[1].axvline(slot * 8 - 0.5, color='grey', linewidth=0.5)
axes[1].set_title('after Flatten: the same 160 numbers in one long row - word 1 is columns 1-8, word 2 is columns 9-16, ...')
plt.tight_layout()
plt.show()

### The reveal: an Embedding layer IS one-hot x a weight matrix
Multiply review 0's one-hot grid (20 x 10,000) by the Embedding layer's weight matrix (10,000 x 8) and you get exactly the 20 x 8 grid the Embedding layer produced. The layer just skips the multiplication and looks the row up - same answer, 10,000 times cheaper.

In [ ]:
embedding_matrix_learned = model.layers[0].get_weights()[0]        # 10,000 words x 8 numbers
by_hand = one_hot @ embedding_matrix_learned                       # (20 x 10,000) @ (10,000 x 8) = 20 x 8
by_layer = model.layers[0](x_train[0:1]).numpy()[0]               # what the Embedding layer gave back
print('one-hot x weights:', by_hand.shape, '| Embedding layer:', by_layer.shape)
print('identical?', np.allclose(by_hand, by_layer))

In [ ]:
# seeing is believing: the multiplication as a picture (only the columns/rows this review uses - the rest are zeros)
used_cols = sorted(set(x_train[0]))
labels_used = []
for col in used_cols:
    labels_used.append(index_word.get(col, '?'))
small_one_hot = one_hot[:, used_cols]                              # 20 x (words used)
small_weights = embedding_matrix_learned[used_cols, :]             # (words used) x 8
lim = np.abs(by_layer).max()
fig, axes = plt.subplots(1, 4, figsize=(17, 6), gridspec_kw={'width_ratios': [len(used_cols), 8, 8, 8]})
axes[0].imshow(small_one_hot, cmap='Greys', vmin=0, vmax=1.4, aspect='auto')
axes[0].set_title('one-hot (20 x ' + str(len(used_cols)) + ')')
axes[0].set_yticks(range(maxlen))
axes[0].set_yticklabels(peek_words[0])
axes[0].set_xticks(range(len(used_cols)))
axes[0].set_xticklabels(labels_used, rotation=70, fontsize=8)
axes[1].imshow(small_weights, cmap='RdBu_r', vmin=-lim, vmax=lim, aspect='auto')
axes[1].set_title('x  embedding weights\n(one row of 8 per word)')
axes[1].set_yticks(range(len(used_cols)))
axes[1].set_yticklabels(labels_used)
axes[2].imshow(by_hand, cmap='RdBu_r', vmin=-lim, vmax=lim, aspect='auto')
axes[2].set_title('=  one-hot x weights\n(20 x 8)')
axes[3].imshow(by_layer, cmap='RdBu_r', vmin=-lim, vmax=lim, aspect='auto')
axes[3].set_title('the Embedding layer\n(20 x 8) - the same!')
for ax in axes[1:]:
    ax.set_xticks(range(8))
    ax.set_xticklabels(range(1, 9))
for ax in axes[2:]:
    ax.set_yticks([])
plt.tight_layout()
plt.show()
print('identical?', np.allclose(by_hand, by_layer))

### What Flatten can't do - and what's next
Flatten scores every word **alone, in its slot**: *not* in slot 4 can't flip *good* in slot 5. A recurrent layer reads the words **in order** and carries a memory, so *not* changes how *good* is read (M5_2c):

![Flatten vs SimpleRNN](https://raw.githubusercontent.com/drdave-teaching/OPIM5509-notebooks/main/Module5/img/m5_flatten_vs_rnn.gif)

## Where this leaves us
* **One-hot** is easy and makes a strong baseline - but it's huge (10,000 numbers per review) and every word is a stranger to every other word.
* **Learned embeddings** fix that: a short, dense vector per word, and words used the same way end up close together. (For vectors learned from 6 billion words instead of our 25,000 reviews, that's GloVe - M5_2a.)
* But **Flatten locks every word to its slot**, so the model never reads the review *in order*. That's the next notebook: recurrent layers (M5_2c).
* Here IMDB came pre-packaged as numbers. In real projects you start from raw text files - the capstone does exactly that, with real posts from Bluesky.